# NSE Smart-Money — end-to-end workflow

**Question:** does institutional ("smart money") accumulation — visible through bulk/block-deal
disclosures, delivery percentages and FII/DII flows — actually precede stronger forward returns
in Indian stocks, or is it folklore?

This notebook runs the full loop: **raw data → SQLite warehouse → features → event studies →
causality tests → predictive models → verdict.**

> Education & research only. Not investment advice.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
pd.set_option("display.width", 160)

## 1. Run the ETL pipeline

`mode="auto"` tries live sources (yfinance + NSE) and falls back to the bundled
deterministic sample dataset if they are unreachable. Use `mode="live"` on a machine
with NSE access.

In [ ]:
from nse_smartmoney.pipeline import run
run(mode="auto")   # or "live" / "sample" 

## 2. Inspect the warehouse

In [ ]:
from nse_smartmoney import storage
for t in ["prices", "fii_dii_flows", "deals", "delivery", "features", "validation"]:
    n = storage.read(f"SELECT COUNT(*) AS n FROM {t}", parse_dates=())["n"][0]
    print(f"{t:<15} {n:>8,} rows")
print("source:", storage.get_meta("source"))

In [ ]:
deals = storage.read("SELECT * FROM deals")
deals.groupby("profile").agg(n_deals=("qty","count"), gross_qty=("qty","sum")).sort_values("n_deals", ascending=False)

## 3. Features

Everything a signal needs to be *tradable* is computed as-of the close of the signal day —
forward returns are the labels.

In [ ]:
features = storage.read("SELECT * FROM features")
features.tail(3).T

## 4. Participant validation — the core event study

For every participant with ≥5 net-buy days on a stock: mean forward return, win rate,
and a one-sided t-test (H₁: mean forward return > 0).

In [ ]:
validation = storage.read("SELECT * FROM validation", parse_dates=())
v10 = validation[validation.horizon == 10].sort_values("p_value")
v10[["symbol","client","profile","n_events","mean_fwd_ret","win_rate","p_value","significant"]].head(10).round(4)

In [ ]:
n_sig, n_all = int(v10.significant.sum()), len(v10)
print(f"{n_sig} of {n_all} participant-symbol pairs significant at 5% "
      f"(≈{n_all*0.05:.1f} expected by chance — mind multiple testing!)")

## 5. Event-study curve — average price path around the top accumulator's buys

In [ ]:
from nse_smartmoney.analysis import event_study_curve
prices = storage.read("SELECT * FROM prices")

top = v10.iloc[0]
ev = deals[(deals.symbol == top.symbol) & (deals.client == top.client) &
           (deals.side == "BUY")][["date","symbol"]].drop_duplicates()
curve = event_study_curve(prices, ev)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(curve.rel_day, curve.mean_cumret*100, marker="o", label="mean")
ax.plot(curve.rel_day, curve.median_cumret*100, ls="--", label="median")
ax.axvline(0, color="k", lw=1); ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("days relative to net-buy event"); ax.set_ylabel("cumulative return (%)")
ax.set_title(f"{top.symbol}: price path around {top.client} net-buy days (n={top.n_events})")
ax.legend(); plt.tight_layout()

## 6. Causality — do flows *lead* returns?

In [ ]:
from nse_smartmoney.analysis import granger_flows, granger_stock_flow
flows = storage.read("SELECT * FROM fii_dii_flows")
print("Aggregate FII/DII flow -> market returns:")
display(granger_flows(features, flows))
print(f"Stock-level smart-money flow -> {top.symbol} returns:")
display(granger_stock_flow(features, top.symbol))

## 7. Predictive models

- **OLS** (HAC/Newey-West robust errors): do the features explain 5-day forward returns in the pooled panel?
- **Logistic regression & random forest**: chronological 70/30 split, predict up/down.

In [ ]:
from nse_smartmoney.modeling import ols_signal_test, classification_suite
model, summary = ols_signal_test(features)
print(f"OLS  R²={model.rsquared:.5f}  n={int(model.nobs):,}")
summary

In [ ]:
results = classification_suite(features)
display(results)
display(results.attrs["importances"])

## 8. Verdict

- **Pooled predictability is ~zero** (R²≈0, ROC-AUC≈0.5): generic "smart-money" features do
  **not** predict short-term returns across the whole universe. That's the honest baseline.
- **Edge lives in pockets**: the event-study layer surfaces *specific participants on specific
  stocks* whose repeated net buying precedes statistically significant positive drift.
- **Multiple testing is the main trap**: with ~70 tests at α=5%, a few "significant" pairs are
  expected by chance. Persistence across time and horizons is the real filter.
- Re-run with `mode="live"` for real NSE data; the notebook and dashboard read the same warehouse.